In [21]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score
)

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

In [2]:
# recreate the synthetic dataset

np.random.seed(42)

n_samples = 1000
d = 2

mu_0 = np.array([-2.0, 0.0])
mu_1 = np.array([2.0, 0.0])

sigma = 1.0

y = np.random.randint(0, 2, size=n_samples)

X = np.zeros((n_samples, d))

for i in range(n_samples):
    if y[i] == 0:
        X[i] = np.random.normal(mu_0, sigma, size=d)
    else:
        X[i] = np.random.normal(mu_1, sigma, size=d)

In [3]:
# create neighborhood dependent labels

k = 10

knn = NearestNeighbors(n_neighbors=k + 1)
knn.fit(X)

distances, indices = knn.kneighbors(X)

neighbor_indices = indices[:, 1:]

neighbor_label_mean = y[neighbor_indices].mean(axis=1)

epsilon = 0.1

y_modified = y.copy()

ambiguous = np.abs(neighbor_label_mean - 0.5) < epsilon

y_modified[ambiguous] = 1 - y_modified[ambiguous]

print("Number of flipped labels:", np.sum(y_modified != y))
print("Percentage flipped:", 100 * np.sum(y_modified != y) / n_samples)

Number of flipped labels: 22
Percentage flipped: 2.2


In [4]:
# construct central points and neighbor sets

X_center = X
X_neighbors = X[neighbor_indices]

print("Central points:", X_center.shape)
print("Neighbor sets:", X_neighbors.shape)
print("Labels:", y_modified.shape)

Central points: (1000, 2)
Neighbor sets: (1000, 10, 2)
Labels: (1000,)


In [5]:
# train-test split

X_center_train, X_center_test, X_neighbors_train, X_neighbors_test, y_train, y_test = train_test_split(
    X_center,
    X_neighbors,
    y_modified,
    test_size=0.2,
    random_state=42,
    stratify=y_modified
)

In [6]:
# convert to PyTorch tensors
X_center_train = torch.tensor(X_center_train, dtype=torch.float32)
X_neighbors_train = torch.tensor(X_neighbors_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)

X_center_test = torch.tensor(X_center_test, dtype=torch.float32)
X_neighbors_test = torch.tensor(X_neighbors_test, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32)

In [7]:
# create data loaders

train_dataset = TensorDataset(
    X_center_train,
    X_neighbors_train,
    y_train
)

test_dataset = TensorDataset(
    X_center_test,
    X_neighbors_test,
    y_test
)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

In [8]:
# multi-head self-attention module
class MultiHeadSelfAttention(nn.Module):

    def __init__(self, input_dim, embed_dim, num_heads):
        super().__init__()

        self.embedding = nn.Linear(input_dim, embed_dim)

        self.attention = nn.MultiheadAttention(
            embed_dim=embed_dim,
            num_heads=num_heads,
            batch_first=True
        )

    def forward(self, x):

        # Convert input features to the embedding dimension
        x = self.embedding(x)

        # Self-attention
        attended, attention_weights = self.attention(
            x, x, x
        )

        return attended, attention_weights

In [11]:
# create test instance

attention = MultiHeadSelfAttention(
    input_dim=2,
    embed_dim=32,
    num_heads=4
)

neighbors_batch = X_neighbors_train[:32]

attended, attention_weights = attention(neighbors_batch)

print("Input shape:", neighbors_batch.shape)
print("Attended shape:", attended.shape)
print("Attention weights shape:", attention_weights.shape)

Input shape: torch.Size([32, 10, 2])
Attended shape: torch.Size([32, 10, 32])
Attention weights shape: torch.Size([32, 10, 10])


In [12]:
# build self-attention block

class SAB(nn.Module):

    def __init__(self, input_dim, embed_dim, num_heads):
        super().__init__()

        self.embedding = nn.Linear(input_dim, embed_dim)

        self.attention = nn.MultiheadAttention(
            embed_dim=embed_dim,
            num_heads=num_heads,
            batch_first=True
        )

        self.norm1 = nn.LayerNorm(embed_dim)

        self.feed_forward = nn.Sequential(
            nn.Linear(embed_dim, embed_dim),
            nn.ReLU(),
            nn.Linear(embed_dim, embed_dim)
        )

        self.norm2 = nn.LayerNorm(embed_dim)

    def forward(self, x):

        # Initial embedding
        x = self.embedding(x)

        # Self-attention
        attended, attention_weights = self.attention(
            x, x, x
        )

        # Residual connection + normalization
        x = self.norm1(x + attended)

        # Feed-forward network
        ff_output = self.feed_forward(x)

        # Second residual connection + normalization
        x = self.norm2(x + ff_output)

        return x, attention_weights

In [13]:
# test the SAB module
sab = SAB(
    input_dim=2,
    embed_dim=32,
    num_heads=4
)

neighbors_batch = X_neighbors_train[:32]

sab_output, sab_attention = sab(neighbors_batch)

print("Input shape:", neighbors_batch.shape)
print("SAB output shape:", sab_output.shape)
print("Attention shape:", sab_attention.shape)

Input shape: torch.Size([32, 10, 2])
SAB output shape: torch.Size([32, 10, 32])
Attention shape: torch.Size([32, 10, 10])


In [14]:
# create PMA class
class PMA(nn.Module):

    def __init__(self, embed_dim, num_heads):
        super().__init__()

        # Learnable seed vector
        self.seed = nn.Parameter(
            torch.randn(1, 1, embed_dim)
        )

        self.attention = nn.MultiheadAttention(
            embed_dim=embed_dim,
            num_heads=num_heads,
            batch_first=True
        )

        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, x):

        # Expand the seed for the whole batch
        seed = self.seed.expand(
            x.size(0), -1, -1
        )

        # Seed attends to the set
        pooled, attention_weights = self.attention(
            seed, x, x
        )

        # Normalize the pooled representation
        pooled = self.norm(pooled)

        return pooled, attention_weights

In [15]:
# test PMA

pma = PMA(
    embed_dim=32,
    num_heads=4
)

pooled, pma_attention = pma(sab_output)

print("SAB output shape:", sab_output.shape)
print("Pooled shape:", pooled.shape)
print("PMA attention shape:", pma_attention.shape)

SAB output shape: torch.Size([32, 10, 32])
Pooled shape: torch.Size([32, 1, 32])
PMA attention shape: torch.Size([32, 1, 10])


In [16]:
# create set transformer
class SetTransformer(nn.Module):

    def __init__(self):
        super().__init__()

        # Self-attention block
        self.sab = SAB(
            input_dim=2,
            embed_dim=32,
            num_heads=4
        )

        # Attention-based pooling
        self.pma = PMA(
            embed_dim=32,
            num_heads=4
        )

        # Final classifier
        self.classifier = nn.Sequential(
            nn.Linear(32 + 2, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, center, neighbors):

        # Process the neighbor set
        set_features, attention_weights = self.sab(neighbors)

        # Pool the set into one representation
        pooled, pma_attention = self.pma(set_features)

        # Remove the set dimension: (batch, 1, 32) -> (batch, 32)
        pooled = pooled.squeeze(1)

        # Combine neighborhood representation with central point
        combined = torch.cat(
            [center, pooled],
            dim=1
        )

        # Classification
        output = self.classifier(combined)

        return output.squeeze(1), attention_weights, pma_attention

In [18]:
# create and inspect the model
model = SetTransformer()

print(model)

# test the model
center_batch = X_center_train[:32]
neighbors_batch = X_neighbors_train[:32]

outputs, attention_weights, pma_attention = model(
    center_batch,
    neighbors_batch
)

print("Output shape:", outputs.shape)
print("SAB attention shape:", attention_weights.shape)
print("PMA attention shape:", pma_attention.shape)

SetTransformer(
  (sab): SAB(
    (embedding): Linear(in_features=2, out_features=32, bias=True)
    (attention): MultiheadAttention(
      (out_proj): NonDynamicallyQuantizableLinear(in_features=32, out_features=32, bias=True)
    )
    (norm1): LayerNorm((32,), eps=1e-05, elementwise_affine=True, bias=True)
    (feed_forward): Sequential(
      (0): Linear(in_features=32, out_features=32, bias=True)
      (1): ReLU()
      (2): Linear(in_features=32, out_features=32, bias=True)
    )
    (norm2): LayerNorm((32,), eps=1e-05, elementwise_affine=True, bias=True)
  )
  (pma): PMA(
    (attention): MultiheadAttention(
      (out_proj): NonDynamicallyQuantizableLinear(in_features=32, out_features=32, bias=True)
    )
    (norm): LayerNorm((32,), eps=1e-05, elementwise_affine=True, bias=True)
  )
  (classifier): Sequential(
    (0): Linear(in_features=34, out_features=32, bias=True)
    (1): ReLU()
    (2): Linear(in_features=32, out_features=1, bias=True)
  )
)
Output shape: torch.Size([32

In [20]:
# set transformer loss and optimizer
criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

# train for 50 epochs
num_epochs = 50

for epoch in range(num_epochs):

    model.train()

    total_loss = 0

    for center_batch, neighbors_batch, labels_batch in train_loader:

        optimizer.zero_grad()

        outputs, _, _ = model(
            center_batch,
            neighbors_batch
        )

        loss = criterion(outputs, labels_batch)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    average_loss = total_loss / len(train_loader)

    if (epoch + 1) % 10 == 0:
        print(
            f"Epoch {epoch + 1}/{num_epochs}, "
            f"Loss: {average_loss:.4f}"
        )

Epoch 10/50, Loss: 0.0600
Epoch 20/50, Loss: 0.0528
Epoch 30/50, Loss: 0.0417
Epoch 40/50, Loss: 0.0411
Epoch 50/50, Loss: 0.0397


In [23]:
# evaluate the model on the test set

model.eval()

all_predictions = []
all_labels = []

with torch.no_grad():

    for center_batch, neighbors_batch, labels_batch in test_loader:

        outputs, _, _ = model(
            center_batch,
            neighbors_batch
        )

        probabilities = torch.sigmoid(outputs)

        predictions = (probabilities >= 0.5).float()

        all_predictions.extend(predictions.numpy())
        all_labels.extend(labels_batch.numpy())

accuracy = accuracy_score(all_labels, all_predictions)

print(f"Accuracy: {accuracy:.4f}")

print("\nClassification Report:")
print(
    classification_report(
        all_labels,
        all_predictions,
        target_names=["Class 0", "Class 1"]
    )
)

print("Confusion Matrix:")
print(confusion_matrix(all_labels, all_predictions))

Accuracy: 0.9600

Classification Report:
              precision    recall  f1-score   support

     Class 0       0.96      0.96      0.96        98
     Class 1       0.96      0.96      0.96       102

    accuracy                           0.96       200
   macro avg       0.96      0.96      0.96       200
weighted avg       0.96      0.96      0.96       200

Confusion Matrix:
[[94  4]
 [ 4 98]]


In [25]:
# permutation invariance test
model.eval()

center = X_center_test[0:1]
neighbors = X_neighbors_test[0:1]

with torch.no_grad():
    original_probability = torch.sigmoid(
        model(center, neighbors)[0]
    ).item()

differences = []

for _ in range(10):

    permutation = torch.randperm(neighbors.shape[1])
    shuffled_neighbors = neighbors[:, permutation, :]

    with torch.no_grad():
        shuffled_probability = torch.sigmoid(
            model(center, shuffled_neighbors)[0]
        ).item()

    differences.append(
        abs(original_probability - shuffled_probability)
    )

print("Original probability:", original_probability)
print("Maximum difference:", max(differences))
print("All differences:", differences)

Original probability: 0.999956488609314
Maximum difference: 0.0
All differences: [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
